In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState) -> JokeState:
    prompt = f"Generate a joke about {state['topic']}."

    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def explain_joke(state: JokeState) -> JokeState:
    prompt = f"Explain the following joke: {state['joke']}."

    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('explain_joke', explain_joke)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'explain_joke')
graph.add_edge('explain_joke', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}

workflow.invoke({'topic': 'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.',
 'explanation': 'A classic play on words. This joke is a pun, which is a type of wordplay that uses multiple meanings of a word to create humor.\n\nIn this joke, "crusty" has a double meaning:\n\n1. A pizza has a crust, which is the outer layer of the bread. So, in a literal sense, a pizza can be "crusty" because of its crust.\n2. "Crusty" can also be used to describe someone who is grumpy, irritable, or in a bad mood. For example, "He\'s been feeling a little crusty all day."\n\nThe joke relies on this double meaning to create a humorous connection between the setup ("Why was the pizza in a bad mood?") and the punchline ("Because it was feeling a little crusty"). The word "crusty" is used to make a clever and unexpected connection between the pizza\'s literal crust and its emotional state, creating a lighthearted and amusing effect.'}

In [9]:
workflow.get_state(config=config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.', 'explanation': 'A classic play on words. This joke is a pun, which is a type of wordplay that uses multiple meanings of a word to create humor.\n\nIn this joke, "crusty" has a double meaning:\n\n1. A pizza has a crust, which is the outer layer of the bread. So, in a literal sense, a pizza can be "crusty" because of its crust.\n2. "Crusty" can also be used to describe someone who is grumpy, irritable, or in a bad mood. For example, "He\'s been feeling a little crusty all day."\n\nThe joke relies on this double meaning to create a humorous connection between the setup ("Why was the pizza in a bad mood?") and the punchline ("Because it was feeling a little crusty"). The word "crusty" is used to make a clever and unexpected connection between the pizza\'s literal crust and its emotional state, creating a lighthearted and amusing effect.'}, next=(), config={'configurab

In [10]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.', 'explanation': 'A classic play on words. This joke is a pun, which is a type of wordplay that uses multiple meanings of a word to create humor.\n\nIn this joke, "crusty" has a double meaning:\n\n1. A pizza has a crust, which is the outer layer of the bread. So, in a literal sense, a pizza can be "crusty" because of its crust.\n2. "Crusty" can also be used to describe someone who is grumpy, irritable, or in a bad mood. For example, "He\'s been feeling a little crusty all day."\n\nThe joke relies on this double meaning to create a humorous connection between the setup ("Why was the pizza in a bad mood?") and the punchline ("Because it was feeling a little crusty"). The word "crusty" is used to make a clever and unexpected connection between the pizza\'s literal crust and its emotional state, creating a lighthearted and amusing effect.'}, next=(), config={'configura